# NDgpu — Phase 4: 3D CMFD-solve bake-off, the payoff (Colab)

Phase 0 found that in **2D** host `scipy` LU beats a device multigrid CMFD solve
across every mesh we run, so CMFD stayed on the host. This notebook tests the
claim that **3D flips that verdict**: on the extruded-prism CMFD diffusion
operator, sparse LU factorisation is O(N^2) work with O(N^{4/3}) fill, so it
blows up in both time and memory, while multigrid stays O(N).

Same "setup + K back-solves" pattern as the 2D bake-off (CMFD reuses one matrix
for many RHS per outer), on the **real** 3D operator (`_dsa_matrix_3d`, the same
sparsity/conditioning class as the drift matrix). We add a **fill / memory**
column -- the 3D-specific evidence -- alongside the strategies:

| strategy | 3D expectation |
|---|---|
| host `scipy` LU | time ~ N^1.8-2, **fill/N grows** (memory blows up) |
| device Jacobi-CG / Neumann-CG | iters ~ N^{1/3}..N^{1/2} |
| **multigrid** (pyamg + GPU V-cycle) | **iters flat, O(N)** |

**GO** (opposite of 2D): if device multigrid's back-solve time grows near-
linearly and overtakes host-LU as N rises -- and if LU fill outgrows memory --
then the CMFD device port is *necessary* for 3D, not optional.

(Cross sections are illustrative placeholders, not predictive.)

In [ ]:
import os
try:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    get_ipython().run_line_magic("pip", f"install -q {zip_name}")
    try:
        import cupy
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q cupy-cuda12x")
    get_ipython().run_line_magic("pip", "install -q pyamg")
    get_ipython().system("nvidia-smi -L")
except ImportError:
    pass

In [ ]:
import time
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import pyamg

from ndgpu import Material
from ndgpu.tri import TriGrid
from ndgpu.tri_sn import TriSNTransportSolver
from ndgpu.linalg import pcg, neumann_preconditioner

try:
    import cupy
    import cupyx.scipy.sparse as csp
    import cupyx.scipy.sparse.linalg as cspla
    HAVE_GPU = cupy.cuda.runtime.getDeviceCount() > 0
except Exception:
    HAVE_GPU = False
print("GPU available:", HAVE_GPU)

QUICK = bool(os.environ.get("NDGPU_QUICK"))
# (in-plane nrc, axial nz). N = nrc^2 * 2 * nz. Bigger sizes need a 16 GB GPU.
SIZES = [(8, 6), (10, 8)] if QUICK else [(8, 6), (10, 8), (12, 10), (14, 12), (16, 14)]
K = 3 if QUICK else 30
RTOL = 1e-6
MAT = Material(diffusion=[1.0], sigma_a=[0.02], nu_sigma_f=[0.025],
               sigma_s=[[0.0]], name="m")


def real_matrix_3d(nrc, nz):
    g = TriGrid(shape=(nrc, nrc, 2, nz), side=3.0, height=3.0 * nz)
    s = TriSNTransportSolver(g, MAT, n_polar=2, n_azi=4, bc="vacuum", engine="lu")
    return s._dsa_matrix_3d(0).tocsr()


def xsync(xp):
    if xp is not np:
        xp.cuda.Stream.null.synchronize()

def xp_csr(A, xp):
    return A if xp is np else csp.csr_matrix(A)

def rhs_batch(N, xp, seed=0):
    B = np.random.default_rng(seed).standard_normal((K, N))
    return B if xp is np else xp.asarray(B)

In [ ]:
# backend-agnostic multigrid V-cycle (pyamg hierarchy on CPU, apply on xp)
def build_mg(A_host, xp, presmooth=2, postsmooth=2):
    ml = pyamg.smoothed_aggregation_solver(A_host, max_coarse=400)
    levels = []
    for L in ml.levels[:-1]:
        A = L.A.tocsr(); d = A.diagonal()
        levels.append(dict(A=xp_csr(A, xp),
                           Dinv=(xp.asarray(1.0/d) if xp is not np else 1.0/d),
                           P=xp_csr(L.P.tocsr(), xp), R=xp_csr(L.R.tocsr(), xp)))
    coarse_lu = spla.factorized(ml.levels[-1].A.tocsc())
    omega = 0.7
    def vcycle(b, i=0):
        if i == len(levels):
            xc = coarse_lu(np.asarray(b) if xp is np else cupy.asnumpy(b))
            return xc if xp is np else xp.asarray(xc)
        lv = levels[i]; x = xp.zeros_like(b)
        for _ in range(presmooth):
            x = x + omega * lv["Dinv"] * (b - lv["A"] @ x)
        ec = vcycle(lv["R"] @ (b - lv["A"] @ x), i + 1)
        x = x + lv["P"] @ ec
        for _ in range(postsmooth):
            x = x + omega * lv["Dinv"] * (b - lv["A"] @ x)
        return x
    return vcycle, len(levels) + 1

def cg_iters(apply_A, b, xp, precond):
    return pcg(apply_A, b, xp.zeros_like(b), None, xp, rtol=RTOL, maxiter=5000,
               precond=precond, raise_on_fail=False)

def strat_host_lu(A):
    t=time.perf_counter(); lu=spla.splu(A.tocsc()); setup=time.perf_counter()-t
    B=rhs_batch(A.shape[0], np)
    t=time.perf_counter()
    for b in B: lu.solve(b)
    fill=lu.L.nnz+lu.U.nnz
    return dict(name="host-LU", setup=setup, ksolve=time.perf_counter()-t,
                iters=0, fill_per_N=fill/A.shape[0])

def strat_iter(name, precond_of, A, xp):
    Ad=xp_csr(A,xp); invd=(xp.asarray(1.0/A.diagonal()) if xp is not np else 1.0/A.diagonal())
    aA=lambda x: Ad@x
    t=time.perf_counter(); pc=precond_of(aA,invd); xsync(xp); setup=time.perf_counter()-t
    B=rhs_batch(A.shape[0],xp); xsync(xp); t=time.perf_counter(); its=[]
    for j in range(K):
        _,it=cg_iters(aA,B[j],xp,pc); its.append(it)
    xsync(xp)
    return dict(name=name,setup=setup,ksolve=time.perf_counter()-t,iters=float(np.mean(its)),fill_per_N=0)

def strat_mg(A, xp):
    Ad=xp_csr(A,xp); aA=lambda x: Ad@x; B=rhs_batch(A.shape[0],xp)
    t=time.perf_counter(); vc,nl=build_mg(A,xp); xsync(xp); setup=time.perf_counter()-t
    xsync(xp); t=time.perf_counter(); its=[]
    for j in range(K):
        _,it=cg_iters(aA,B[j],xp,lambda r: vc(r)); its.append(it)
    xsync(xp)
    return dict(name="MG",setup=setup,ksolve=time.perf_counter()-t,iters=float(np.mean(its)),fill_per_N=0)

In [ ]:
def bake(xp, tag):
    print(f"\n==== {tag}: setup + {K} back-solves, real 3D prism CMFD operator ====")
    print(f"{'nrc':>4} {'nz':>3} {'N':>7} {'setup[s]':>9} {'Ksolve[s]':>10} "
          f"{'per_solve[ms]':>13} {'iters':>6} {'fill/N':>7}")
    data={}
    for nrc,nz in SIZES:
        A=real_matrix_3d(nrc,nz); N=A.shape[0]
        res=[]
        if xp is np:
            res.append(strat_host_lu(A))
        res.append(strat_iter("jacobi-CG", lambda aA,invd:(lambda r:invd*r), A, xp))
        res.append(strat_iter("neumann-CG", lambda aA,invd:neumann_preconditioner(aA,invd,6), A, xp))
        res.append(strat_mg(A, xp))
        for r in res:
            r["N"]=N; data.setdefault(r["name"],[]).append(r)
            fp=f"{r['fill_per_N']:>7.0f}" if r['fill_per_N'] else f"{'-':>7}"
            print(f"{nrc:>4} {nz:>3} {N:>7} {r['setup']:>9.3f} {r['ksolve']:>10.3f} "
                  f"{1e3*r['ksolve']/K:>13.2f} {r['iters']:>6.1f} {fp}")
    return data

cpu_data=bake(np,"CPU")
gpu_data=bake(cupy,"GPU") if HAVE_GPU else {}

print("\nPower-law  time ~ N^p  (2D LU was ~1.5 and won; 3D LU ~1.8-2 = blows up):")
for name,rows in (gpu_data if HAVE_GPU else cpu_data).items():
    Ns=np.array([r["N"] for r in rows],float); ys=np.array([r["ksolve"] for r in rows],float)
    if len(Ns)>=2 and (ys>0).all():
        print(f"  {name:>10}: p = {np.polyfit(np.log(Ns),np.log(ys),1)[0]:.2f}")
print("host-LU fill/N (grows with N in 3D = O(N^4/3) memory; ~flat in 2D):")
for r in cpu_data.get("host-LU",[]):
    print(f"  N={r['N']:>7}  fill/N={r['fill_per_N']:.0f}")

In [ ]:
import matplotlib.pyplot as plt
src = gpu_data if HAVE_GPU else cpu_data
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True)
for name,rows in src.items():
    ax[0].plot([r["N"] for r in rows],[r["ksolve"] for r in rows],"o-",label=name)
ax[0].set(xlabel="N (prisms)",ylabel="K back-solves [s]",xscale="log",yscale="log",
          title=f"{'GPU' if HAVE_GPU else 'CPU'}: solve time vs N"); ax[0].legend(fontsize=8); ax[0].grid(True,which="both",alpha=0.3)
for name,rows in src.items():
    if rows[0]["iters"]>0:
        ax[1].plot([r["N"] for r in rows],[r["iters"] for r in rows],"o-",label=name)
ax[1].set(xlabel="N",ylabel="avg iters",xscale="log",title="iterations vs N (flat = O(N))"); ax[1].legend(fontsize=8); ax[1].grid(True,which="both",alpha=0.3)
lu=cpu_data.get("host-LU",[])
if lu:
    ax[2].plot([r["N"] for r in lu],[r["fill_per_N"] for r in lu],"s-",color="C3")
    ax[2].set(xlabel="N",ylabel="LU fill / N",xscale="log",title="host-LU fill blowup (3D)"); ax[2].grid(True,which="both",alpha=0.3)
plt.show()

## Reading the results — the verdict flips

* **`fill/N` climbs with N** (rightmost panel): the sparse-LU factor grows
  super-linearly in 3D (O(N^{4/3})), so host-LU runs out of memory on a real
  core -- the wall 2D never hit.
* **`p` (LU) ~ 1.8-2** vs the 2D bake-off's ~1.5: LU time is heading to O(N^2).
* **MG iterations stay flat** and its back-solve time trends to **p ~ 1** (O(N)),
  so on the GPU it overtakes host-LU as N grows -- the opposite of the 2D result.

If that holds on your GPU, **GO**: the CMFD device port (multigrid) is required
for 3D. The backend-agnostic V-cycle benchmarked here is the same code the port
would wire into `_cmfd_factor_3d` / a device-resident `_cmfd_power` (Phases that
follow), so this bake-off both justifies and de-risks it.